# 2 · Claude Code I: AI as a Coding Copilot

**The question of this session:** the market prices Apple at **$309.35 a share** — **$4.51 trillion** in total. Is that justified? You will answer the way an analyst does: value Apple against its nine largest technology peers, using their own SEC filings, and publish the tool that does it to your GitHub.

**In this notebook you will:**

- Compute what the market charges per unit of earnings, across the peer group
- Value Apple at its peers' multiples, end to end, and compare with $309
- Turn that single answer into an honest range with Monte Carlo simulation
- Publish your first finance project to GitHub

## The working rhythm
With Claude Code you are the analyst in charge; the model is a fast assistant:

1. **Ask small.** One function, one fix, one chart at a time.
2. **Read before you run.** Accept only changes you can explain.
3. **Commit at every green moment.** Small commits make each step reversible.

**How to use Claude Code in this notebook:** select an exercise's docstring, press `Option+K` (Mac) / `Alt+K` (Windows): the file and lines land in the ✱ panel: then ask *"implement this"*. Read the diff it proposes before accepting.

**Pandas in one paragraph:** a `DataFrame` is the analyst's table. `pd.read_csv` loads it; columns are vectors, so `df["a"] / df["b"]` computes a whole ratio column at once; you don't memorize pandas: you *specify* what you want and *verify* what you get.

In [ ]:
import sys, os, json
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
# If you pulled course updates while this kernel was running, pick them up here
# (a no-op on a fresh kernel; saves a restart otherwise):
import importlib
for _n in [n for n in list(sys.modules) if n.startswith("toolkit")]:
    importlib.reload(sys.modules[_n])
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ImportError:
    pass
HAS_KEY = bool(os.environ.get("ANTHROPIC_API_KEY"))
print(f"repo root: {ROOT}")
print(f"API key:   {'configured' if HAS_KEY else 'NOT SET - cells that call Claude will be skipped'}")

## Part A · The question, and the data

$309.35 is a **price**: it emerged from Friday's trading. Today we compute an independent estimate of Apple's **worth**, by asking what the prices of its peers imply. The method is **comparable-company analysis**.

The data: ten companies, fundamentals from each one's **latest annual SEC filing**, prices at Friday's close. Four definitions carry the session:

- **EBITDA** (earnings before interest, taxes, depreciation and amortization) **≈ operating income + D&A** — what the operations earn, before financing and accounting choices.
- **Market capitalization = shares × share price** — the price of the equity.
- **Enterprise value (EV) = market cap + debt − cash** — the price of the whole business.
- **EV/EBITDA = enterprise value ÷ EBITDA** — the whole business per unit of what it earns. **A multiple is a rating, not a price**: it becomes an estimate of worth only when a *peer's* rating is applied to Apple's own earnings — which is exactly what Part B does.

Load the dataset:

In [ ]:
import numpy as np
import pandas as pd
pd.options.display.float_format = "{:,.1f}".format

companies = pd.read_csv(ROOT / "session-02-coding-copilot" / "data" / "tech_financials.csv")
print(f"{len(companies)} companies, prices as of {companies['price_asof'].iloc[0]}")
companies   # the full cast: Apple and its nine potential peers

### Exercise 1: know the peer group

Before pricing Apple off its peers, know who the peers are: how fast each grows, and what margin it earns. These two columns are also what you will hover on in the chart, and what Session 4's screening engine filters on.

Add five columns: `revenue_growth_1y` (vs prior), `revenue_cagr_2y` (two-year compound growth: `(rev/rev_prior2)**0.5 - 1`), `ebitda_m`, `op_margin`, `ebitda_margin`.

In [ ]:
def add_growth_and_margins(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
### START CODE HERE ###
    df["revenue_growth_1y"] = df[None] / df[None] - 1
    df["revenue_cagr_2y"] = (df[None] / df[None]) ** None - 1    # exponent for a TWO-year CAGR?
    df["ebitda_m"] = df[None] + df[None]                         # EBITDA = which two columns?
    df["op_margin"] = df[None] / df["revenue_m"]
    df["ebitda_margin"] = df[None] / df["revenue_m"]
### END CODE HERE ###
    return df

companies = add_growth_and_margins(companies)
companies[["ticker", "revenue_growth_1y", "ebitda_margin"]].round(3)

In [ ]:
# ✅ self-check: run me
r = companies.set_index("ticker")
assert "revenue_growth_1y" in companies and "ebitda_margin" in companies, "missing columns"
assert abs(r.loc["AAPL", "revenue_growth_1y"] - (r.loc["AAPL", "revenue_m"] / r.loc["AAPL", "revenue_prior_m"] - 1)) < 1e-9
assert r.loc["INTC", "op_margin"] < 0, "Intel's operating margin should be negative - don't 'fix' real data"
assert (companies["revenue_cagr_2y"].abs() < 1.5).all(), "CAGR out of range - check the exponent (2-year: **0.5)"
print("All checks passed ✅")

### Exercise 2: what the market charges per unit of earnings

For each company: build the enterprise-value bridge, then three multiples — **EV/EBITDA** (the anchor), **EV/Sales** (EV ÷ revenue, the fallback when EBITDA is negative), and **P/E** (market cap ÷ net income, the equity per unit of net profit).

One company will misbehave by design: **Intel's net income is negative**, and a multiple over a negative denominator is meaningless — so its P/E must print as `NaN`, reported as **n.m.** (not meaningful), the professional convention your check enforces.

In [ ]:
def add_multiples(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
### START CODE HERE ###
    df["mcap_m"] = df[None] * df[None]                            # market cap = shares x ?
    df["ev_m"] = df["mcap_m"] + df[None] - df[None]               # EV = mcap + debt - cash
    df["ev_ebitda"] = np.where(df["ebitda_m"] > 0, df[None] / df[None], np.nan)
    df["ev_sales"] = df["ev_m"] / df[None]
    df["pe"] = np.where(df[None] > 0, df["mcap_m"] / df[None], np.nan)
### END CODE HERE ###
    return df

companies = add_multiples(companies)
companies[["ticker", "ev_ebitda", "ev_sales", "pe"]].round(1)

In [ ]:
# ✅ self-check: run me
r = companies.set_index("ticker")
assert np.isnan(r.loc["INTC", "pe"]), "Intel lost money: P/E must be NaN (n.m.), never negative"
assert 5 < r.loc["AAPL", "ev_ebitda"] < 60, "AAPL EV/EBITDA looks wrong - check EV = mcap + debt - cash"
assert (companies["ev_ebitda"].dropna() > 0).all(), "no negative multiples allowed"
print("All checks passed ✅")
print()

### Exercise 3: the summary an analyst could actually use

One table, sorted by EV/EBITDA, with a **MEDIAN** row at the bottom. That median is the anchor: it is the multiple the deterministic valuation of Apple applies in Part B.

Return `ticker, revenue_m, revenue_growth_1y, ebitda_margin, ev_ebitda, ev_sales, pe`, sorted by `ev_ebitda` ascending (NaN last), plus a final `MEDIAN` row with column medians.

In [ ]:
def build_summary(df: pd.DataFrame) -> pd.DataFrame:
    cols = ["ticker", "revenue_m", "revenue_growth_1y", "ebitda_margin", "ev_ebitda", "ev_sales", "pe"]
### START CODE HERE ###
    summary = df[cols].sort_values(None, na_position="last")      # sort by which multiple?
    median = summary.drop(columns="ticker").median(numeric_only=True)
    summary = pd.concat([summary, pd.DataFrame([{"ticker": None, **median.to_dict()}])],
                        ignore_index=True)                        # label for the final row?
### END CODE HERE ###
    return summary

summary = build_summary(companies)
summary.round(2)

In [ ]:
# ✅ self-check: run me
assert summary.iloc[-1]["ticker"] == "MEDIAN", "last row must be the MEDIAN"
assert len(summary) == len(companies) + 1
v = summary["ev_ebitda"].dropna().iloc[:-1]
assert (v.values == sorted(v.values)).all(), "sort by ev_ebitda ascending"
print("All checks passed ✅")

OUTD = ROOT / "outputs"; OUTD.mkdir(exist_ok=True)
summary.to_csv(OUTD / "comps_summary.csv", index=False)
print("Saved outputs/comps_summary.csv - this file goes in your portfolio.")

### The chart: multiples at a glance

A table answers precise questions; a chart shows who stands out. This one is **interactive**: hover a bar for the company's growth and margin, drag to zoom, double-click to reset. (Plotly draws interactive charts; seaborn and matplotlib, which the course also installs, draw static ones.)

In [ ]:
# plotly draws the chart; its notebook renderer needs nbformat. Install either
# if missing (happens on environments created before they joined the course).
import importlib, subprocess, sys
for _pkg in ("plotly", "nbformat"):
    try:
        importlib.import_module(_pkg)
    except ModuleNotFoundError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg], check=True)
import plotly.express as px

plot_df = companies.dropna(subset=["ev_ebitda"]).sort_values("ev_ebitda")
fig = px.bar(
    plot_df, x="ticker", y="ev_ebitda",
    hover_data={"revenue_growth_1y": ":.1%", "ebitda_margin": ":.1%", "ev_ebitda": ":.1f"},
    labels={"ev_ebitda": "EV/EBITDA (x)", "ticker": ""},
    title="EV/EBITDA by company (a company with negative EBITDA would show n.m. and be excluded)",
)
fig.add_hline(y=plot_df["ev_ebitda"].median(), line_dash="dash", annotation_text="median")
fig.show()

## Part B · Value Apple

**What we compare, exactly.** Share prices cannot be compared across companies: price per share depends on how many shares a company happens to have (a 2-for-1 stock split halves the price and changes nothing about the business). What *is* comparable is a **rating** — price per dollar of earnings — because the arbitrary share count cancels out. The real comparison of this session is therefore Apple's rating against its peers' ratings. The chain below only converts the peers' rating back into dollars per Apple share, so the verdict can be read against the one number everyone knows: the actual market price.

First, the chain that turns a peer multiple into a share price — this is how comparable-company analysis prices a company:

```
1. peer EV/EBITDA multiple × Apple's EBITDA     =  implied enterprise value
2. implied EV − debt + cash                     =  implied equity value
3. implied equity value ÷ shares                =  implied share price
```

Step 2 is your own Exercise 2 bridge, walked backwards: `EV = market cap + debt − cash`, so recovering the equity from an EV means subtracting the debt and adding the cash back.

**Choosing the peers — and why two companies are excluded.** A peer's multiple is only a fair rating if its *earnings are normal*, because a multiple is price ÷ earnings: when the earnings are temporarily depressed or the price reflects a very different future, the ratio stops meaning "what the market charges per unit of earnings." Two companies in our table fail that test today:

- **Intel** is in a loss-making turnaround: its earnings are depressed, so its multiple is inflated by a shrunken denominator. It rates Intel's crisis, not Apple.
- **AMD** trades at 183× trailing EBITDA: the market is pricing earnings it expects *in the future*, far above what AMD earns today. That is a bet on AMD's growth, not a rating anyone would apply to Apple.

That leaves **seven peers**: Microsoft, Alphabet, Meta, Amazon, NVIDIA, Oracle and Salesforce. The cell below values Apple the **deterministic** way — the *median* of those seven ratings, walked through the three steps, every intermediate printed. Read it line by line: this is a company being valued, end to end.

In [ ]:
# The deterministic version: one multiple (the peer median), one price. GIVEN - read it.
PEERS = ["MSFT", "GOOGL", "META", "AMZN", "NVDA", "ORCL", "CRM"]   # chosen above, reasons stated

target = companies.set_index("ticker").loc["AAPL"]
peer_multiples = companies.set_index("ticker").loc[PEERS, "ev_ebitda"]
print("The seven peer ratings (EV/EBITDA):")
print(peer_multiples.round(1).to_string(), "\n")

median_multiple = peer_multiples.median()
implied_ev     = median_multiple * target["ebitda_m"]                       # step 1
implied_equity = implied_ev - target["total_debt_m"] + target["cash_m"]     # step 2: bridge, backwards
implied_price  = implied_equity / target["shares_m"]                        # step 3

print(f"peer median EV/EBITDA        : {median_multiple:8.1f}x")
print(f"x Apple's EBITDA             : {target['ebitda_m']:8,.0f}m")
print(f"= implied enterprise value   : {implied_ev:8,.0f}m")
print(f"- debt + cash                : {implied_equity:8,.0f}m  (implied equity value)")
print(f"/ shares outstanding         : {target['shares_m']:8,.0f}m")
print(f"= IMPLIED SHARE PRICE        : ${implied_price:7.2f}")
print(f"  actual market price        : ${target['price_usd']:7.2f}")
print()
print(f"In plain words: at a TYPICAL peer rating, Apple would be worth about ${implied_price:.0f}.")
print(f"The market pays ${target['price_usd']:.0f} - roughly {target['price_usd']/implied_price-1:.0%} more. Is that premium justified?")
print("One number cannot say. The next exercise asks all seven ratings instead of the median.")

### Exercise 4: from a point to a range

The deterministic answer dodged one question: *which* peer multiple is the right one? Nobody knows — and **Monte Carlo simulation** is the honest response: draw many random samples from the uncertain input (here, the peer multiples), run each through the same chain, and read the *distribution* of implied prices instead of a single number.

**The benefit, stated plainly: the range turns a crude verdict into a precise one.** The point estimate ($191 vs $309) invites a verdict — "Apple trades roughly 60% above peer value." How much that verdict really says depends on how much the seven peers disagree among themselves, and only the distribution can tell you. You are about to find out.

Two of the gaps ask you to write step 2, the inverted bridge. One clarification, because it is a common confusion: **each multiple prices its own denominator** — EV/EBITDA multiplies EBITDA; EV/Sales would multiply revenue.

In [ ]:
rng = np.random.default_rng(7)   # seeded, so every run of this cell gives the same result

def implied_price_range(df: pd.DataFrame, ticker: str, peer_tickers=None, n_sims: int = 10_000):
    """Sample peer EV/EBITDA ratings and convert each into an implied share price."""
    row = df.set_index("ticker").loc[ticker]
    if peer_tickers is None:                       # default: everyone else with a meaningful multiple
        peer_multiples = df.loc[df["ticker"] != ticker, "ev_ebitda"].dropna().values
    else:
        peer_multiples = df.set_index("ticker").loc[peer_tickers, "ev_ebitda"].dropna().values
### START CODE HERE ###
    sampled = rng.choice(peer_multiples, size=n_sims, replace=None)  # sample WITH replacement?
    implied_ev = sampled * row[None]                                 # the rating prices which figure?
    implied_mcap = implied_ev - row[None] + row[None]                # invert the EV bridge
    implied_price = implied_mcap / row["shares_m"]
### END CODE HERE ###
    return implied_price

In [ ]:
# ✅ self-check: run me (offline, deterministic). One peer multiple of exactly 10x,
# EBITDA 100, debt 40, cash 20, 10 shares -> implied price must be exactly 98.
toy = pd.DataFrame({"ticker": ["TARGET", "PEER"], "ebitda_m": [100.0, 50.0],
                    "total_debt_m": [40.0, 0.0], "cash_m": [20.0, 0.0],
                    "shares_m": [10.0, 1.0], "ev_ebitda": [float("nan"), 10.0]})
prices = implied_price_range(toy, "TARGET", n_sims=100)
assert abs(prices.mean() - 98.0) < 1e-9, "check the EV bridge inversion: (10*100 - 40 + 20) / 10 = 98"
print("All checks passed ✅  The bridge inverts correctly. Now the whole table:")

In [ ]:
NORMAL = [t for t in companies["ticker"] if t not in ("INTC", "AMD")]   # ratings with normal earnings

rows = []
for t in companies["ticker"]:
    peer_list = [x for x in NORMAL if x != t]
    prices = implied_price_range(companies, t, peer_tickers=peer_list)
    actual = companies.set_index("ticker").loc[t, "price_usd"]
    rows.append({"ticker": t, "p10": np.percentile(prices, 10),
                 "p50": np.percentile(prices, 50), "p90": np.percentile(prices, 90),
                 "actual": actual, "share_above": (prices > actual).mean()})
mc = pd.DataFrame(rows)
print(mc.round(2).to_string(index=False))

import importlib, subprocess, sys
for _pkg in ("plotly", "nbformat"):      # self-heal, in case this cell runs on a fresh kernel
    try:
        importlib.import_module(_pkg)
    except ModuleNotFoundError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg], check=True)
import plotly.graph_objects as go
fig = go.Figure()
fig.add_bar(x=mc["ticker"], y=mc["p90"] - mc["p10"], base=mc["p10"],
            name="peer-implied range (P10-P90)", marker_color="lightsteelblue",
            hovertemplate="P10 %{base:$,.0f} - P90 %{y:$,.0f}<extra></extra>")
fig.add_scatter(x=mc["ticker"], y=mc["actual"], mode="markers", name="actual price",
                marker=dict(color="firebrick", size=10, symbol="diamond"))
fig.add_scatter(x=mc["ticker"], y=mc["p50"], mode="markers", name="median implied",
                marker=dict(color="steelblue", size=8))
fig.update_layout(title="What normal peer ratings imply each share is worth (10,000 simulations each)",
                  yaxis_title="share price (USD)", yaxis_type="log")
fig.show()

*Reading the chart, plainly.* Each bar answers one question for one company: **"what would this share be worth at its peers' ratings?"** The bottom of the bar is a pessimistic peer rating, the top an optimistic one, the blue dot the middle. The red diamond is what the market actually charges.

**Apple's row, spelled out — this is the session's finding:**

1. At a *typical* peer rating (the median, 19.5×), Apple would be worth about **$191**. The market pays **$309** — a premium of roughly **60%**.
2. Could *any* peer rating justify $309? Almost none: with today's data only **NVIDIA's** rating (39×) implies more, about $383. Alphabet's, the next highest, implies about $278. Overall, only about **14% of the sampled ratings** justify the market price.
3. So the honest conclusion, in one sentence: **the market prices Apple as if it were nearly the best company in this peer group.** Not "Apple is overpriced" — the method cannot say that — but "Apple is priced for near-best-in-group performance."
4. What comparable-company analysis *cannot* tell you is whether that confidence is deserved: whether Apple's growth and quality genuinely merit an almost-NVIDIA rating. **That question needs the company's own cash flows — tomorrow's DCF.**

Two more rows worth a glance, because the same tool exposes them: **AMD and Intel** show diamonds far above their bars. For AMD, the market is paying for future earnings that trailing numbers cannot see; for Intel, the market is betting the turnaround restores earnings. In both cases the gap between diamond and bar is not an error — it is the size of the bet the market is making, made visible.

**What Monte Carlo bought you today:** the single median invited a verdict ("62% overpriced"); the full range of ratings turned it into a precise, defensible statement ("priced for near-best-in-group; only one peer's rating implies more"). A range with a stated assumption is an analysis; a point without one is an opinion.

## Part C · Publish to GitHub
Your comparable-company analysis tool is a project. In the **terminal** (not this notebook):

```bash
git init && git add . && git commit -m "Comps tool: first working version, AAPL multiple hand-verified"
gh repo create my-finance-toolkit --private --source . --push
```

(No `gh`? github.com → New repo → follow "push an existing repository". Cheatsheet: `cheatsheets/git-github-for-finance.md`.)

## Deliverable checklist

- [ ] All three ✅ checks green, with at least one exercise implemented via Claude Code (`Option/Alt+K` on the docstring)
- [ ] `outputs/comps_summary.csv` exists; Intel shows NaN P/E
- [ ] Repo pushed to GitHub with ≥2 commits
- [ ] Stretch: scatter `revenue_growth_1y` vs `ev_ebitda`: is growth priced in?

**Next:** `03-debugging-earnings.ipynb`: a valuation model that is wrong on purpose.